# Stock Data Analysis Dashboard



In [1]:

%pip -q install yfinance pandas numpy plotly scikit-learn statsmodels scipy

In [9]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
from IPython.display import display, Markdown

TICKER = "MSFT"   # Change this ticker
PERIOD = "1y"     # 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y
INTERVAL = "1d"   # 1d, 1wk, 1mo

print(f"Downloading {TICKER} ({PERIOD}, {INTERVAL})...")
data = pd.DataFrame()
try:
    data = yf.download(TICKER, period=PERIOD, interval=INTERVAL,
                       auto_adjust=False, progress=False, threads=False)
except Exception as exc:
    print("Download error:", exc)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
data = data.dropna(how="all")

if data.empty or "Close" not in data.columns:
    print("Live download did not return usable data. Upload a historical-price CSV instead.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No CSV uploaded. Re-run this cell after uploading a file.")
    filename = next(iter(uploaded))
    raw = pd.read_csv(filename)
    raw.columns = [str(c).strip() for c in raw.columns]
    # Match common column names case-insensitively
    rename = {c: c.title() for c in raw.columns if c.lower() in ["date", "open", "high", "low", "close", "volume", "adj close"]}
    raw = raw.rename(columns=rename)
    if "Date" not in raw.columns or "Close" not in raw.columns:
        raise ValueError("CSV must include at least Date and Close columns.")
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    raw = raw.dropna(subset=["Date", "Close"]).sort_values("Date").set_index("Date")
    for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
        if col in raw.columns:
            raw[col] = pd.to_numeric(raw[col].astype(str).str.replace(",", "", regex=False), errors="coerce")
    data = raw
    TICKER = filename.rsplit(".", 1)[0]

# Ensure the index is datetime and close prices are numeric
data.index = pd.to_datetime(data.index, errors="coerce")
data = data.loc[~data.index.isna()].sort_index()
data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
data = data.dropna(subset=["Close"])
if len(data) < 5:
    raise ValueError("Not enough valid rows. Please provide at least 5 dated price records.")

close = data["Close"].squeeze().astype(float).dropna()
returns = close.pct_change().dropna()
print(f"Loaded {len(data):,} rows for {TICKER}: {data.index.min().date()} to {data.index.max().date()}")
display(data.tail())

Loaded 252 rows for MSFT: 2025-10-22 to 2026-10-22


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-10-16,529.049988,529.049988,543.820007,524.429993,541.309998,24845700
2026-10-19,533.520020,533.520020,535.570007,528.059998,528.700012,17994900
2026-10-20,531.869995,531.869995,534.210022,525.840027,533.909973,14576000
2026-10-21,532.950012,532.950012,533.280029,525.020020,529.440002,13909900
2026-10-22,525.664978,525.664978,533.596008,525.099976,532.700012,2679145


## 2. Summary metrics and price charts

In [10]:
previous_close = float(close.iloc[-2]) if len(close) > 1 else float(close.iloc[-1])
last_close = float(close.iloc[-1])
change_pct = ((last_close / previous_close) - 1) * 100 if previous_close else 0
total_return = ((close.iloc[-1] / close.iloc[0]) - 1) * 100 if len(close) > 1 else 0
annual_vol = returns.std() * np.sqrt(252) * 100 if len(returns) > 1 else np.nan
max_drawdown = ((close / close.cummax()) - 1).min() * 100

summary = pd.DataFrame({"Metric": ["Latest close", "Previous close", "Latest change (%)", "Total return over selected period (%)", "Annualized volatility (%)", "Maximum drawdown (%)", "Number of observations"],
                        "Value": [last_close, previous_close, change_pct, total_return, annual_vol, max_drawdown, len(close)]})
display(summary.style.format({"Value": "{:,.3f}"}))

chart = pd.DataFrame({"Close": close, "SMA20": close.rolling(20).mean(), "SMA50": close.rolling(50).mean()})
fig = px.line(chart, title=f"{TICKER}: closing price and moving averages", labels={"value":"Price", "index":"Date"})
fig.show()

if all(c in data.columns for c in ["Open", "High", "Low", "Close"]):
    candle = go.Figure(data=[go.Candlestick(x=data.index, open=data["Open"], high=data["High"], low=data["Low"], close=data["Close"])])
    candle.update_layout(title=f"{TICKER}: candlestick chart", xaxis_rangeslider_visible=False)
    candle.show()

if "Volume" in data.columns:
    px.bar(x=data.index, y=data["Volume"], title=f"{TICKER}: trading volume", labels={"x":"Date", "y":"Volume"}).show()
px.histogram(returns * 100, nbins=40, title="Daily return distribution", labels={"value":"Daily return (%)"}).show()

,Metric,Value
0,Latest close,525.665
1,Previous close,532.950
2,Latest change (%),-1.367
3,Total return over selected period (%),0.985
4,Annualized volatility (%),32.941
5,Maximum drawdown (%),-34.911
6,Number of observations,252.000


## 3. Descriptive statistics and risk analysis

In [11]:
descriptive = close.describe(percentiles=[.05, .25, .5, .75, .95]).to_frame("Close price")
descriptive.loc["Skewness"] = close.skew()
descriptive.loc["Kurtosis"] = close.kurtosis()
display(descriptive)

risk = pd.DataFrame({"Metric": ["Mean daily return (%)", "Daily return standard deviation (%)", "Annualized volatility (%)", "Historical VaR 95% daily (%)", "Historical CVaR 95% daily (%)", "Maximum drawdown (%)"],
                     "Value": [returns.mean()*100, returns.std()*100, annual_vol,
                               returns.quantile(0.05)*100 if len(returns) else np.nan,
                               returns[returns <= returns.quantile(0.05)].mean()*100 if len(returns) else np.nan,
                               max_drawdown]})
display(risk.style.format({"Value":"{:,.4f}"}))

if len(returns) >= 8:
    jb = stats.jarque_bera(returns)
    print(f"Jarque–Bera normality test: statistic={jb.statistic:.4f}, p-value={jb.pvalue:.6g}")
if len(close) >= 20:
    try:
        adf = adfuller(close, autolag="AIC")
        print(f"ADF test on close prices: statistic={adf[0]:.4f}, p-value={adf[1]:.6g}")
        print("ADF null hypothesis: the series has a unit root. A small p-value provides evidence against that null.")
    except Exception as exc:
        print("ADF test unavailable:", exc)
rolling_vol = returns.rolling(20).std() * np.sqrt(252) * 100
px.line(x=rolling_vol.index, y=rolling_vol, title="20-period rolling annualized volatility", labels={"x":"Date", "y":"Volatility (%)"}).show()

if len(returns) > 1:
    print("Return correlation with previous period:", round(returns.corr(returns.shift(1)), 4))

,Close price
count,252.000000
mean,448.406289
std,52.650716
min,352.829987
5%,372.816998
25%,400.437508
50%,451.120010
75%,493.782501
95%,529.162488
max,543.450012


,Metric,Value
0,Mean daily return (%),0.0250
1,Daily return standard deviation (%),2.0751
2,Annualized volatility (%),32.9408
3,Historical VaR 95% daily (%),-2.8039
4,Historical CVaR 95% daily (%),-4.0493
5,Maximum drawdown (%),-34.9106


Jarque–Bera normality test: statistic=1921.7389, p-value=0
ADF test on close prices: statistic=-1.4549, p-value=0.555698
ADF null hypothesis: the series has a unit root. A small p-value provides evidence against that null.


Return correlation with previous period: 0.0933


## 4. Technical indicators

In [12]:
tech = data.copy()
tech["Close"] = close
tech["SMA20"] = close.rolling(20).mean()
tech["SMA50"] = close.rolling(50).mean()
tech["EMA12"] = close.ewm(span=12, adjust=False).mean()
tech["EMA26"] = close.ewm(span=26, adjust=False).mean()
tech["MACD"] = tech["EMA12"] - tech["EMA26"]
tech["MACD_signal"] = tech["MACD"].ewm(span=9, adjust=False).mean()
delta = close.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = -delta.clip(upper=0).rolling(14).mean()
rs = gain / loss.replace(0, np.nan)
tech["RSI14"] = 100 - (100 / (1 + rs))
mid = close.rolling(20).mean()
sd = close.rolling(20).std()
tech["BB_upper"] = mid + 2 * sd
tech["BB_lower"] = mid - 2 * sd

band_fig = go.Figure()
band_fig.add_trace(go.Scatter(x=tech.index, y=close, name="Close"))
band_fig.add_trace(go.Scatter(x=tech.index, y=tech["BB_upper"], name="Bollinger upper", line=dict(dash="dot")))
band_fig.add_trace(go.Scatter(x=tech.index, y=tech["BB_lower"], name="Bollinger lower", line=dict(dash="dot")))
band_fig.update_layout(title="Bollinger Bands")
band_fig.show()
px.line(x=tech.index, y=tech["RSI14"], title="Relative Strength Index (14)", labels={"x":"Date", "y":"RSI"}).show()
px.line(tech[["MACD", "MACD_signal"]], title="MACD", labels={"index":"Date", "value":"MACD"}).show()
display(tech[[c for c in ["Close","SMA20","SMA50","RSI14","MACD","MACD_signal","BB_upper","BB_lower"] if c in tech.columns]].tail(15))

Price,Close,SMA20,SMA50,RSI14,MACD,MACD_signal,BB_upper,BB_lower
Date,,,,,,,,
2026-10-02,517.530029,501.672000,487.724200,57.825420,7.804419,7.445329,518.702579,484.641421
2026-10-05,525.179993,502.945999,490.593799,68.268216,8.630326,7.682328,522.914229,482.977769
2026-10-06,529.299988,504.713498,493.397799,76.315776,9.507712,8.047405,527.401739,482.025257
2026-10-07,529.760010,506.618999,496.125999,73.848896,10.123471,8.462618,531.024116,482.213881
2026-10-08,522.609985,508.127498,498.767399,70.507866,9.920163,8.754127,532.572076,483.682920
2026-10-09,535.070007,510.099498,500.446799,72.330469,10.641788,9.131659,536.577844,483.621152
2026-10-12,541.150024,511.886499,501.975399,77.878245,11.570905,9.619508,541.652303,482.120695
2026-10-13,543.450012,514.203000,503.091400,77.795035,12.350458,10.165698,546.253608,482.152392
2026-10-14,533.690002,516.372501,503.909000,71.235126,12.041897,10.540938,547.470484,485.274518


## 5. Baseline machine-learning models

These models use chronological splitting to avoid randomly mixing older and newer observations. Their results are historical test metrics, not reliable guarantees of future prices.

In [6]:
frame = pd.DataFrame({"Close": close})
frame["Lag1"] = frame["Close"].shift(1)
frame["Lag2"] = frame["Close"].shift(2)
frame["Lag3"] = frame["Close"].shift(3)
frame["MA5"] = frame["Close"].rolling(5).mean()
frame["MA10"] = frame["Close"].rolling(10).mean()
frame = frame.dropna()
features = ["Lag1", "Lag2", "Lag3", "MA5", "MA10"]

if len(frame) >= 40:
    split = max(int(len(frame) * 0.8), 20)
    train, test = frame.iloc[:split], frame.iloc[split:]
    results = []
    predictions = pd.DataFrame(index=test.index)
    for name, model in [("Linear Regression", LinearRegression()),
                        ("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=2))]:
        model.fit(train[features], train["Close"])
        pred = model.predict(test[features])
        predictions[name] = pred
        results.append({"Model": name,
                        "MAE": mean_absolute_error(test["Close"], pred),
                        "RMSE": np.sqrt(mean_squared_error(test["Close"], pred)),
                        "R2": r2_score(test["Close"], pred) if len(test) >= 2 else np.nan})
    display(pd.DataFrame(results).set_index("Model"))
    eval_df = pd.DataFrame({"Actual": test["Close"]}).join(predictions)
    px.line(eval_df, title="Model predictions versus actual close price").show()
else:
    print("At least 40 usable rows are recommended for the ML baseline models; choose a longer period or upload more data.")

,MAE,RMSE,R2
Model,,,
Linear Regression,18.390236,26.699720,0.955815
Random Forest,35.491921,47.566872,0.859761


## 7. Export analyzed data

This downloads the historical data plus technical indicator columns as a CSV file.

In [ ]:
export_df = tech.copy()
export_df.index.name = "Date"
export_name = f"{str(TICKER).replace('/', '_')}_stock_analysis.csv"
export_df.to_csv(export_name)
print("Created:", export_name)
files.download(export_name)